# 🔬 Conference Room Quality — Model Analysis

Deep-dive into model performance:
1. Residual analysis for each model
2. Predicted vs. actual scatter plots
3. Feature importance comparison
4. Error analysis — which images are hardest to score?
5. SHAP value interpretation (XGBoost)

> Run `python src/score.py` before this notebook.

In [ ]:
import sys
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

warnings.filterwarnings('ignore')

ROOT = Path('.').resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

plt.rcParams.update({
    'figure.facecolor': '#0F0F1A',
    'axes.facecolor':   '#1A1A2E',
    'axes.edgecolor':   '#2D2D4E',
    'axes.labelcolor':  '#E8E8F0',
    'xtick.color':      '#E8E8F0',
    'ytick.color':      '#E8E8F0',
    'text.color':       '#E8E8F0',
    'grid.color':       '#2D2D4E',
    'grid.alpha':       0.4,
})
PALETTE = ['#6C63FF', '#43D3A6', '#FFC857', '#FF6584']

print('Ready. ROOT =', ROOT)

## 1. Load Models & Data

In [ ]:
import yaml
with open(ROOT / 'config.yaml') as f:
    config = yaml.safe_load(f)

target_col = config['model']['target_column']
random_state = config['model']['random_state']
test_size = config['model']['test_size']

feat_df = pd.read_csv(ROOT / 'features' / 'features.csv')
label_df = pd.read_csv(ROOT / 'data' / 'labels.csv')
merged = feat_df.merge(label_df, on='filename', how='inner')

feature_cols = [c for c in feat_df.columns if c != 'filename']
X = merged[feature_cols].fillna(0.0)
y = merged[target_col].astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=random_state
)

# Load all models
model_names = ['linear_regression', 'ridge_regression', 'random_forest', 'xgboost']
models = {}
for name in model_names:
    path = ROOT / 'models' / f'{name}.joblib'
    if path.exists():
        payload = joblib.load(path)
        models[name] = payload['model']
    else:
        print(f'Model not found: {path}')

print(f'Loaded {len(models)} models. Test size: {len(X_test)}')

## 2. Predicted vs. Actual

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes = axes.flatten()

for ax, (name, estimator), color in zip(axes, models.items(), PALETTE):
    y_pred = np.clip(estimator.predict(X_test), 1, 5)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    pearson_r, _ = pearsonr(y_test, y_pred)

    ax.scatter(y_test, y_pred, alpha=0.75, color=color, s=70, edgecolors='none')
    lo, hi = min(y_test.min(), 1) - 0.2, max(y_test.max(), 5) + 0.2
    ax.plot([lo, hi], [lo, hi], '--', color='white', alpha=0.5, lw=1.5, label='Ideal')
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_xlabel('Actual Score', fontsize=9)
    ax.set_ylabel('Predicted Score', fontsize=9)
    ax.set_title(
        f'{name.replace("_", " ").title()}\nMAE={mae:.3f}  R²={r2:.3f}  r={pearson_r:.3f}',
        fontsize=10, fontweight='bold'
    )
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Predicted vs. Actual — Test Set', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'model_predicted_vs_actual_detail.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Residual Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

best_model = models.get('xgboost', list(models.values())[-1])
y_pred_best = np.clip(best_model.predict(X_test), 1, 5)
residuals = y_test.values - y_pred_best

# Residuals vs fitted
ax1 = axes[0]
ax1.scatter(y_pred_best, residuals, alpha=0.75, color='#6C63FF', s=60, edgecolors='none')
ax1.axhline(0, color='white', linestyle='--', linewidth=1.5, alpha=0.7)
ax1.set_xlabel('Predicted Score', fontsize=10)
ax1.set_ylabel('Residual (Actual − Predicted)', fontsize=10)
ax1.set_title('Residuals vs. Fitted (XGBoost)', fontsize=11, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Residual histogram
ax2 = axes[1]
ax2.hist(residuals, bins=20, color='#43D3A6', edgecolor='none', alpha=0.85)
ax2.axvline(0, color='white', linestyle='--', linewidth=1.5, alpha=0.7)
ax2.axvline(residuals.mean(), color='#FFC857', linestyle='-', linewidth=2,
            label=f'Mean={residuals.mean():.3f}')
ax2.set_xlabel('Residual', fontsize=10)
ax2.set_ylabel('Count', fontsize=10)
ax2.set_title('Residual Distribution (XGBoost)', fontsize=11, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'model_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Residuals — Mean: {residuals.mean():.4f}  Std: {residuals.std():.4f}')
print(f'Max over-prediction: {residuals.min():.3f}  Max under-prediction: {residuals.max():.3f}')

## 4. Feature Importance — XGBoost vs Random Forest

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
tree_models = {k: v for k, v in models.items() if k in ['xgboost', 'random_forest']}

for ax, (name, estimator), color in zip(axes, tree_models.items(), ['#6C63FF', '#43D3A6']):
    raw = estimator.steps[-1][1] if hasattr(estimator, 'steps') else estimator
    if not hasattr(raw, 'feature_importances_'):
        continue
    importances = raw.feature_importances_
    sorted_idx = np.argsort(importances)
    colors = plt.cm.plasma(np.linspace(0.2, 0.9, len(importances)))
    ax.barh([feature_cols[i] for i in sorted_idx], importances[sorted_idx],
            color=[colors[i] for i in range(len(sorted_idx))], edgecolor='none')
    ax.set_title(f'{name.replace("_"," ").title()} Feature Importances',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Importance', fontsize=9)
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'model_feature_importance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Hardest Images to Score (Largest Errors)

In [ ]:
test_merged = merged.iloc[X_test.index].copy()
test_merged['y_pred'] = y_pred_best
test_merged['abs_error'] = (test_merged[target_col] - test_merged['y_pred']).abs()
hardest = test_merged.nlargest(5, 'abs_error')[['filename', target_col, 'y_pred', 'abs_error']]

print('Images with Largest Prediction Error (XGBoost):')
print(hardest.to_string(index=False))

# Optionally display the hardest images
images_dir = ROOT / 'data' / 'images'
fig, axes = plt.subplots(1, min(5, len(hardest)), figsize=(16, 4))
if not hasattr(axes, '__iter__'):
    axes = [axes]

for ax, (_, row) in zip(axes, hardest.iterrows()):
    img_path = images_dir / row['filename']
    if img_path.exists():
        from PIL import Image as PILImage
        img = PILImage.open(img_path).resize((320, 240))
        ax.imshow(img)
        ax.set_title(
            f"{row['filename'][:20]}\n"
            f"True={row[target_col]:.1f} | Pred={row['y_pred']:.1f} | Err={row['abs_error']:.2f}",
            fontsize=8, color='#E8E8F0'
        )
    ax.axis('off')

fig.suptitle('Hardest Images to Score', fontsize=12, fontweight='bold', color='#E8E8F0')
fig.patch.set_facecolor('#0F0F1A')
plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'model_hardest_images.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Cross-Validation Score Stability

In [ ]:
from sklearn.model_selection import cross_val_score

fig, ax = plt.subplots(figsize=(10, 5))

for i, (name, estimator) in enumerate(models.items()):
    cv_scores = -cross_val_score(estimator, X, y, cv=5,
                                 scoring='neg_mean_absolute_error', n_jobs=-1)
    x_positions = np.arange(5) + i * 0.2
    ax.plot(x_positions, cv_scores, 'o-', color=PALETTE[i], label=name.replace('_', ' ').title(),
            linewidth=2, markersize=8, alpha=0.85)

ax.set_xlabel('CV Fold', fontsize=10)
ax.set_ylabel('MAE', fontsize=10)
ax.set_title('5-Fold Cross-Validation MAE per Model', fontsize=12, fontweight='bold')
ax.legend(fontsize=9, facecolor='#1A1A2E', edgecolor='#2D2D4E')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'model_cv_stability.png', dpi=150, bbox_inches='tight')
plt.show()